# Lite-DPAS V4: NMSE 학습 + DPAS 지식 증류

이 노트북은 V3의 8×8 latent 구조에 **16×16 encoder spectral gate**를 추가하고, 검증 지표와 일치하는 **샘플별 NMSE 손실**, plateau 기반 학습률 감소, 선택적 **원본 DPAS 교사 증류**를 적용합니다.

- 입력/출력: `[B, 2, 32, 32]`, 실수부·허수부 정규화 CSI
- CR=1/4 기본 feedback codeword: 512 real values
- 증류를 켜려면 학습된 DPAS checkpoint와 DPAS-CsiNet 저장소 경로가 필요합니다.
- `RESUME_FROM`에는 V4 checkpoint만 지정하세요. V2/V3 checkpoint는 구조가 달라 재개할 수 없습니다.
- 훈련 중에는 train/validation만 사용하며 test split은 열지 않습니다.


In [ ]:
from pathlib import Path
import json, sys, time, shutil
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

# Colab에서는 프로젝트 폴더를 Drive/MyDrive/CSI_SLLM_RESEARCH에 두세요.
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/CSI_SLLM_RESEARCH')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / 'src').is_dir() and (PROJECT_ROOT / 'CSI_SLLM_RESEARCH' / 'src').is_dir():
        PROJECT_ROOT = PROJECT_ROOT / 'CSI_SLLM_RESEARCH'
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('CSI_SLLM_RESEARCH 프로젝트 폴더를 찾지 못했습니다. PROJECT_ROOT를 직접 지정하세요.')
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / 'Data'
if IN_COLAB:
    # MAT 파일은 세션 로컬에서 읽고, checkpoint와 로그는 Drive의 PROJECT_ROOT에 저장합니다.
    LOCAL_DATA_DIR = Path('/content/cost2100_indoor')
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for name in ['DATA_Htrainin.mat', 'DATA_Hvalin.mat']:
        src, dst = DATA_DIR / name, LOCAL_DATA_DIR / name
        if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
            shutil.copy2(src, dst)
    DATA_DIR = LOCAL_DATA_DIR
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Project:', PROJECT_ROOT)
print('Data:', DATA_DIR)
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

In [ ]:
# 실험 설정: 첫 실행은 1 epoch로 확인한 다음 전체 학습으로 바꾸세요.
COMPRESSION_RATIO = 4       # 지원값: 4, 16, 32, 64
EPOCHS = 1000               # 예비 확인은 1, 본 학습은 1000
BATCH_SIZE = 200            # GPU 메모리 부족 시 64 또는 32
SEED = 42
RUN_TAG = 'v4_nmse_baseline_01'      # 기존 v2_fresh_01 결과 보존; 재실험은 새 태그 사용
SAVE_EVERY = 100            # epoch 주기 checkpoint 저장
RESUME_FROM = None          # 예: Path('/content/drive/MyDrive/.../checkpoint_epoch_0100.pt')
RUN_TRAINING = False        # 학습을 시작하려면 True로 변경
PROFILE_SAMPLES = 512      # validation 프로파일에 사용할 표본 수

assert COMPRESSION_RATIO in (4, 16, 32, 64)
ENCODED_DIM = 2048 // COMPRESSION_RATIO
required = ['DATA_Htrainin.mat', 'DATA_Hvalin.mat']
missing = [name for name in required if not (DATA_DIR / name).is_file()]
assert not missing, f'Data 폴더에 파일이 없습니다: {missing}'
print(f'CR=1/{COMPRESSION_RATIO} | latent={ENCODED_DIM} real values | epochs={EPOCHS} | batch={BATCH_SIZE}')
print('Train/validation 파일 확인 완료. Test 파일은 이 노트북에서 열지 않습니다.')
# 교사 증류: 사용 가능한 DPAS 학습 checkpoint가 있을 때만 켜세요.
USE_DISTILLATION = False
DPAS_REPO_DIR = Path('/content/DPAS-CsiNet') if IN_COLAB else PROJECT_ROOT.parent / 'DPAS-CsiNet'
DPAS_TEACHER_CHECKPOINT = None  # 예: Path('/content/drive/MyDrive/.../dpas_best.pth')
DISTILL_WEIGHT = 0.25
LOSS_MODE = 'nmse'             # 검증 sample-wise NMSE와 정렬
SCHEDULER_PATIENCE = 25        # plateau 시 학습률을 절반으로 낮춤


## 1. V4 모델 동작 확인

무작위 입력으로 shape, 출력 범위, 역전파를 확인합니다. 성능 측정은 학습 뒤 validation 프로파일 단계에서 확인합니다.


In [ ]:
from src.models.lite_dpas import LiteDPASCsiNetV4, count_parameters
from src.models.csinet import CsiNet

model = LiteDPASCsiNetV4(encoded_dim=ENCODED_DIM).to(DEVICE)
sample = torch.rand(2, 2, 32, 32, device=DEVICE)
output = model(sample)
assert output.shape == sample.shape
assert torch.isfinite(output).all()
assert output.min() >= 0 and output.max() <= 1
((output - sample) ** 2).mean().backward()

baseline_params = count_parameters(CsiNet(encoded_dim=ENCODED_DIM))
lite_params = count_parameters(model)
print('Forward/backward: OK')
print('Input/output:', tuple(sample.shape), tuple(output.shape))
print(f'Parameters: Lite-DPAS {lite_params:,} | CsiNet port {baseline_params:,}')
print(f'Parameter change: {(1 - lite_params / baseline_params) * 100:.1f}% fewer than this CsiNet port')

## 2. 학습 및 선택적 교사 증류

먼저 `RUN_TRAINING=True`, `EPOCHS=1`로 경로를 확인하고, 본 학습에서는 1000 epoch 등 원하는 값으로 바꾸세요. 증류는 실제 DPAS checkpoint 경로가 준비되었을 때 `USE_DISTILLATION=True`로 켭니다. 이 실험은 새 `RUN_TAG`로 저장합니다.


In [ ]:
from src.train_baseline import train_model

if USE_DISTILLATION:
    if DPAS_TEACHER_CHECKPOINT is None:
        raise ValueError('USE_DISTILLATION=True이면 DPAS_TEACHER_CHECKPOINT를 지정하세요.')
    if not Path(DPAS_TEACHER_CHECKPOINT).is_file():
        raise FileNotFoundError(f'DPAS teacher checkpoint를 찾지 못했습니다: {DPAS_TEACHER_CHECKPOINT}')
    if not DPAS_REPO_DIR.is_dir():
        import subprocess
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/OhayoRINK/DPAS-CsiNet.git', str(DPAS_REPO_DIR)], check=True)

if RUN_TRAINING:
    CHECKPOINT = train_model(
        'litedpas_v4', PROJECT_ROOT, DATA_DIR,
        epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED, encoded_dim=ENCODED_DIM,
        run_tag=RUN_TAG, save_every=SAVE_EVERY, resume_from=RESUME_FROM,
        loss_mode=LOSS_MODE, scheduler_patience=SCHEDULER_PATIENCE,
        teacher_checkpoint=DPAS_TEACHER_CHECKPOINT if USE_DISTILLATION else None,
        teacher_repo=DPAS_REPO_DIR if USE_DISTILLATION else None,
        distill_weight=DISTILL_WEIGHT,
    )
    print('Best validation checkpoint:', CHECKPOINT)
else:
    CHECKPOINT = PROJECT_ROOT / 'runs' / f'litedpas_v4_indoor_cr{COMPRESSION_RATIO}_{EPOCHS}ep_{RUN_TAG}' / 'best.pt'
    print('학습은 실행하지 않았습니다. 설정 셀에서 RUN_TRAINING=True로 바꾸세요.')
    print('예상 checkpoint:', CHECKPOINT)


## 3. Validation 성능·비용 프로파일

학습 checkpoint를 불러와 validation 일부의 NMSE, 파라미터 수, MAC 근사치, batch-1 지연을 확인합니다. MAC 근사치는 FFT 비용을 완전히 포함하지 않으므로 실제 지연과 함께 해석하세요.


In [ ]:
from src.cost2100 import COST2100Dataset
from src.metrics import sample_nmse_linear
from src.profile_lite_dpas import approximate_macs, measure_latency_ms

if not CHECKPOINT.is_file():
    print('Checkpoint가 없습니다. 먼저 학습 셀을 실행하세요.')
else:
    saved = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
    profiled_model = LiteDPASCsiNetV4(encoded_dim=int(saved.get('encoded_dim', ENCODED_DIM)))
    profiled_model.load_state_dict(saved['model'])
    profiled_model = profiled_model.to(DEVICE).eval()
    validation = COST2100Dataset(DATA_DIR / 'DATA_Hvalin.mat')
    n = min(PROFILE_SAMPLES, len(validation))
    val_loader = DataLoader(Subset(validation, range(n)), batch_size=64, shuffle=False, num_workers=0)
    ratios = []
    with torch.inference_mode():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            ratios.append(sample_nmse_linear(batch, profiled_model(batch)).cpu())
    val_nmse_db = float(10 * np.log10(max(torch.cat(ratios).mean().item(), 1e-12)))
    one = torch.zeros(1, 2, 32, 32, device=DEVICE)
    latency = measure_latency_ms(profiled_model, one, warmup=20, repeats=100)
    macs = approximate_macs(profiled_model, one)
    report = {
        'model': 'Lite-DPAS-V4', 'compression_ratio': f"1/{2048 // int(saved.get('encoded_dim', ENCODED_DIM))}",
        'validation_samples': n, 'validation_nmse_db': val_nmse_db,
        'parameters': count_parameters(profiled_model),
        'approx_conv_linear_macs_batch1': macs,
        'batch1_latency_ms': latency, 'device': str(DEVICE), 'test_split_used': False,
        'mac_note': 'FFT, activation, normalization and interpolation excluded',
    }
    display(report)
    report_path = CHECKPOINT.parent / 'profile_validation.json'
    report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
    print('Saved:', report_path)

## 실험 순서

1. `USE_DISTILLATION=False`로 V4 + NMSE 손실 + plateau scheduler의 기준 결과를 얻습니다.
2. 기준 학습이 끝나면 `RUN_TAG`를 새 이름으로 바꾸고 `USE_DISTILLATION=True`로 바꾼 뒤 DPAS teacher checkpoint를 지정합니다.
3. `DISTILL_WEIGHT`를 0.1, 0.25, 0.5로 바꿔 한 번에 하나씩 비교합니다.
4. `history.csv`의 validation NMSE와 learning rate, `summary.json`의 설정을 함께 보관합니다.
5. V3와 V4, 증류 전후 결과는 같은 데이터 split·feedback dimension·평가 metric으로 비교합니다.

교사 checkpoint는 원본 DPAS-CsiNet의 학습 결과여야 합니다. 모델 정의 파일은 `DPAS_REPO_DIR/models/dpas_csinet_v5.py` 또는 `dpas_csinet.py`에서 자동으로 찾습니다.
